# Datapoint: Chip Component Spend — v2 (record-store edition)

Same datapoint and model as `chip_component_spend.ipynb`, but **every input is read from the
record datastore** under `data/records/` (plain JSON records + blobs), discovered by scanning and
filtering on `tags`/`type` — no shared store library, no `datasources_old/`. The forecast must
reproduce v1 exactly. The final section writes the forecast back into the store as records.

§1 History · §2 Datasources · §3 Model · §4 Forecast (+ sanity check vs v1) · §5 Visualization · §6 Persist.

## 0. Datasource manifest (agent-curated)
I scanned `data/records/` and selected the specific records this datapoint needs — listed explicitly
below. The notebook reads each named file directly; it does **not** glob the store or filter by tag.

This is the intended workflow for any metric: the agent asks *"do I have the sources I need?"*, scans
the store, and (a) if a source exists, adds its path to this manifest; (b) if not, finds a source,
creates the record, then lists it here. The notebook is just the explicit set of records it imports.

In [1]:
import json, pathlib, datetime
from statistics import mean, pstdev
import polars as pl
import plotly.express as px
import plotly.graph_objects as go

COMPONENTS = ["Memory", "Logic", "Packaging", "Auxiliary"]
COLORS = {
    "Memory": "#4fa8a0",
    "Logic": "#e0a44a",
    "Packaging": "#4a5fd0",
    "Auxiliary": "#d65a9a",
}
REPO = pathlib.Path("..")  # notebook runs from notebooks/


def read_record(relative_path):
    return json.loads((REPO / relative_path).read_text())


# History (processed_data card) selected for this datapoint:
HISTORY_RECORD = "data/records/processed_data/pd_chip-spend-history_7d52.json"

# Datasource records selected for this datapoint (each path found by scanning data/records/):
DATASOURCE_RECORDS = [
    "data/records/news/2026/06/nws_tsmc-q1-rev_5e80.json",
    "data/records/news/2026/06/nws_tsmc-q2-guide_9495.json",
    "data/records/news/2026/06/nws_sk-hynix-q1_71b0.json",
    "data/records/news/2026/06/nws_micron-q2fy26_0bd4.json",
    "data/records/news/2026/06/nws_samsung-q1_7338.json",
    "data/records/news/2026/06/nws_nvidia-q1fy27_20cf.json",
    "data/records/news/2026/06/nws_astera-labs-q1_2691.json",
    "data/records/news/2026/06/nws_monolithic-power-q1_38bd.json",
    "data/records/news/2026/06/nws_broadcom-q2fy26_6318.json",
    "data/records/news/2026/06/nws_tsmc-hpc-platform_b357.json",
    "data/records/news/2026/06/nws_nvidia-blackwell-shipments_e53f.json",
    "data/records/news/2026/06/nws_ase-adv-packaging_5050.json",
    "data/records/news/2026/06/nws_trendforce-dram-q1_7636.json",
    "data/records/news/2026/06/nws_trendforce-dram-q2_bc2a.json",
    "data/records/news/2026/06/nws_trendforce-hbm-bit-supply_ca9b.json",
    "data/records/news/2026/06/nws_trendforce-hbm-bit-demand_1610.json",
    "data/records/news/2026/06/nws_trendforce-cowos-capacity_5c9a.json",
    "data/records/news/2026/06/nws_optical-transceivers_b301.json",
    "data/records/news/2026/06/nws_murata-mlcc_754b.json",
    "data/records/news/2026/06/nws_morganstanley-vr200-bom_b8e6.json",
    "data/records/facts/2026/06/fct_amkor-10q_bbe3.json",
    "data/records/facts/2026/06/fct_korea-motie-customs_23ee.json",
    "data/records/facts/2026/06/fct_fred-ipg3344s_1612.json",
]

history_card = read_record(HISTORY_RECORD)
datasource_records = [read_record(path) for path in DATASOURCE_RECORDS]
print(f"history record: {history_card['id']}")
print(
    f"{len(datasource_records)} datasource records imported by explicit path "
    f"({sum(r['type']=='news' for r in datasource_records)} news + "
    f"{sum(r['type']=='facts' for r in datasource_records)} facts)"
)

history record: pd_chip-spend-history_7d52
23 datasource records imported by explicit path (20 news + 3 facts)


## 1. Datapoint History
Read the history record's blob CSV (the record was selected by path in §0).

In [2]:
history = pl.read_csv(REPO / history_card["body"]["blob_path"])
print("history blob:", history_card["body"]["blob_path"])
history

history blob: data/blobs/chip_component_spend_history_quarterly.csv


Quarter,Memory,Logic,Packaging,Auxiliary,Total
str,f64,f64,f64,f64,f64
"""Q1 2024""",1.6773,0.4579,0.6109,0.4875,3.2336
"""Q2 2024""",2.3065,0.5808,0.7745,0.6061,4.2679
"""Q3 2024""",3.3393,0.8227,1.073,0.7389,5.9739
"""Q4 2024""",4.8054,1.2289,1.5148,0.9523,8.5014
"""Q1 2025""",5.5342,1.4082,1.6949,1.2167,9.854
"""Q2 2025""",6.2846,1.5063,1.7979,1.1499,10.7387
"""Q3 2025""",8.7238,1.8867,2.2416,1.3816,14.2337
"""Q4 2025""",11.0007,2.2421,2.5715,1.5319,17.3462


In [3]:
history_long = history.unpivot(
    index="Quarter", on=COMPONENTS, variable_name="Component", value_name="spend"
)
figure = px.bar(
    history_long,
    x="Quarter",
    y="spend",
    color="Component",
    category_orders={"Component": COMPONENTS},
    color_discrete_map=COLORS,
    labels={"spend": "USD billions"},
    title="Chip component spend - history (NVIDIA/AMD/Google/Amazon)",
)
figure.update_layout(height=440)
figure

## 2. Datasources for datapoint
Build the source lookup from the records imported in §0: `source_value(name, key)` indexes each record's `body.figures` (the figures an agent transcribed from the source). `DATASOURCES` carries the `name/components/role/targets` the engine needs for anchor counting.

In [4]:
DATASOURCES = [
    {
        "name": r["body"]["name"],
        "component": r["body"]["components"],
        "role": r["body"]["role"],
        "targets": r["body"]["targets"],
    }
    for r in datasource_records
]
DATASOURCE_FIGURES = {
    r["body"]["name"]: r["body"]["figures"] for r in datasource_records
}


def source_value(name, key):
    return DATASOURCE_FIGURES[name][key]


datasources_table = pl.DataFrame(
    [
        {
            "type": r["type"],
            "name": r["body"]["name"],
            "component": ",".join(r["body"]["components"]),
            "role": r["body"]["role"],
            "confidence": r["confidence"],
            "source": r["source"],
        }
        for r in datasource_records
    ]
)
print(
    f"{len(DATASOURCES)} datasource records "
    f"({sum(r['type']=='news' for r in datasource_records)} news + "
    f"{sum(r['type']=='facts' for r in datasource_records)} facts)"
)
with pl.Config(tbl_rows=30, fmt_str_lengths=60, tbl_width_chars=200):
    print(datasources_table.sort(["type", "name"]))

23 datasource records (20 news + 3 facts)
shape: (23, 6)
┌───────┬────────────────────────────────────────────────────────┬────────────────────────────┬─────────────┬────────────┬───────────────────────────────────────────────────────────────┐
│ type  ┆ name                                                   ┆ component                  ┆ role        ┆ confidence ┆ source                                                        │
│ ---   ┆ ---                                                    ┆ ---                        ┆ ---         ┆ ---        ┆ ---                                                           │
│ str   ┆ str                                                    ┆ str                        ┆ str         ┆ f64        ┆ str                                                           │
╞═══════╪════════════════════════════════════════════════════════╪════════════════════════════╪═════════════╪════════════╪═══════════════════════════════════════════════════════════════╡
│ facts 

## 3. Clear model for datapoint
**3a — Derive scalars from the record figures** (identical derivations to v1; each scalar indexes a record's `body.figures`).

In [5]:
def quarterly_cagr(start, end, quarters):
    return (end / start) ** (1 / quarters) - 1


def range_midpoint(value_range):
    return sum(value_range) / len(value_range)


BAND = 0.30  # MODELING: relative half-width for low/high when the source gives a point (not a range)


def band(base, low=None, high=None):
    return {
        "low": round(low if low is not None else base * (1 - BAND), 4),
        "base": round(base, 4),
        "high": round(high if high is not None else base * (1 + BAND), 4),
    }


cowos_qoq = quarterly_cagr(
    range_midpoint(
        source_value("TrendForce - CoWoS capacity", "capacity_end2025_wpm_range")
    ),
    range_midpoint(
        source_value("TrendForce - CoWoS capacity", "capacity_end2026_wpm_range")
    ),
    4,
)
optical_qoq = quarterly_cagr(
    source_value("Optical transceivers (InnoLight/Coherent)", "market_2024_busd"),
    source_value("Optical transceivers (InnoLight/Coherent)", "market_2026_busd"),
    8,
)
nvidia_demand_qoq = (1 + source_value("NVIDIA", "dc_yoy_approx")) ** (1 / 4) - 1
aux_supply_qoq = range_midpoint(
    [
        source_value("Astera Labs", "revenue_qoq"),
        source_value("Monolithic Power", "ai_revenue_qoq"),
    ]
)
tsmc_q2_qoq = (
    range_midpoint(
        [
            source_value("TSMC (Q2 guide)", "q2_guide_low_busd"),
            source_value("TSMC (Q2 guide)", "q2_guide_high_busd"),
        ]
    )
    / source_value("TSMC (Q2 guide)", "q1_rev_busd")
    - 1
)


def qoq_from_yoy(yoy):
    return (1 + yoy) ** (1 / 4) - 1


hbm_demand_source = "TrendForce - HBM bit demand & AI server shipments 2026"
volume_qoq = {
    "low": round(
        qoq_from_yoy(source_value(hbm_demand_source, "ai_server_shipments_yoy")), 4
    ),
    "base": round(
        qoq_from_yoy(source_value(hbm_demand_source, "hbm_bit_demand_yoy")), 4
    ),
    "high": round(
        qoq_from_yoy(source_value(hbm_demand_source, "asic_hbm_demand_yoy")), 4
    ),
}

# FRED is read from the pinned facts record (deterministic), not fetched live as in v1.
ip_acceleration = source_value("FRED IPG3344S (US semis IP)", "ip_yoy") - source_value(
    "FRED IPG3344S (US semis IP)", "ip_yoy_prev_q"
)
korea_exports_hot = source_value("Korea MOTIE / customs", "semi_exports_yoy") > 1.0
macro_tilt = max(
    -0.05, min(0.05, ip_acceleration + (0.02 if korea_exports_hot else -0.02))
)
macro_scaler = {
    "low": round(1 + macro_tilt - 0.03, 3),
    "base": round(1 + macro_tilt, 3),
    "high": round(1 + macro_tilt + 0.03, 3),
}

trendforce_q1, trendforce_q2 = (
    "TrendForce - DRAM/NAND contract price (Q1)",
    "TrendForce - DRAM/NAND contract price (Q2)",
)

TARGETS = {
    "Q1 2026": dict(
        kind="nowcast",
        price_qoq={
            "low": source_value(trendforce_q1, "prior_dram_estimate_range")[0],
            "base": source_value(trendforce_q1, "server_dram_qoq"),
            "high": source_value(trendforce_q1, "conventional_dram_qoq_range")[1],
        },
        volume_qoq=volume_qoq,
        supply_qoq={
            "Memory": band(source_value("SK hynix", "revenue_qoq")),
            "Logic": band(
                source_value("TSMC HPC platform (earnings coverage)", "hpc_qoq")
            ),
            "Packaging": band(cowos_qoq),
            "Auxiliary": band(aux_supply_qoq),
        },
        demand_qoq={component: band(nvidia_demand_qoq) for component in COMPONENTS},
        macro_scaler=macro_scaler,
        analyst_qoq=None,
        weights={
            "supply": 0.30,
            "price": 0.30,
            "trend": 0.20,
            "macro": 0.12,
            "demand": 0.08,
        },
    ),
    "Q2 2026": dict(
        kind="forecast",
        price_qoq={
            "low": source_value(trendforce_q2, "dram_contract_qoq_range")[0],
            "base": range_midpoint(
                source_value(trendforce_q2, "dram_contract_qoq_range")
            ),
            "high": source_value(trendforce_q2, "dram_contract_qoq_range")[1],
        },
        volume_qoq=volume_qoq,
        supply_qoq={
            "Memory": band(
                range_midpoint(source_value(trendforce_q2, "dram_contract_qoq_range"))
            ),
            "Logic": band(
                tsmc_q2_qoq,
                source_value("TSMC (Q2 guide)", "q2_guide_low_busd")
                / source_value("TSMC (Q2 guide)", "q1_rev_busd")
                - 1,
                source_value("TSMC (Q2 guide)", "q2_guide_high_busd")
                / source_value("TSMC (Q2 guide)", "q1_rev_busd")
                - 1,
            ),
            "Packaging": band(cowos_qoq),
            "Auxiliary": band(aux_supply_qoq),
        },
        demand_qoq={component: band(nvidia_demand_qoq) for component in COMPONENTS},
        macro_scaler=macro_scaler,
        analyst_qoq={
            "Memory": {"low": 0.25, "base": 0.40, "high": 0.60},
            "Logic": {"low": 0.05, "base": 0.10, "high": 0.16},
            "Packaging": {"low": 0.06, "base": 0.12, "high": 0.20},
            "Auxiliary": {"low": 0.05, "base": 0.10, "high": 0.18},
        },
        weights={
            "supply": 0.26,
            "price": 0.26,
            "trend": 0.18,
            "analyst": 0.12,
            "macro": 0.10,
            "demand": 0.08,
        },
    ),
}
print("Derived base scalars (from records):")
print(
    f"  volume_qoq = {volume_qoq}; macro_scaler = {macro_scaler} (FRED momentum {ip_acceleration:+.3f} + Korea)"
)
print(
    f"  cowos_qoq={cowos_qoq:.3f}  tsmc_q2_qoq={tsmc_q2_qoq:.3f}  nvidia_demand_qoq={nvidia_demand_qoq:.3f}  aux_supply_qoq={aux_supply_qoq:.3f}"
)

Derived base scalars (from records):
  volume_qoq = {'low': 0.0466, 'base': 0.1419, 'high': 0.1583}; macro_scaler = {'low': 0.969, 'base': 0.999, 'high': 1.029} (FRED momentum -0.021 + Korea)
  cowos_qoq=0.133  tsmc_q2_qoq=0.103  nvidia_demand_qoq=0.189  aux_supply_qoq=0.133


**3b — The engine** (verbatim from v1): build estimate families, reconcile, price×volume for Memory, confidence from anchors + dispersion, data-driven band; Q2 chains off Q1.

In [6]:
ANCHOR_ROLES = {"hard", "hard-price", "analyst"}
BASE_HW, CV_REF = 0.12, 0.35
CONF_HIGH = dict(min_anchors=2, max_cv=0.20)
CONF_MED = dict(min_anchors=1, max_cv=0.30)


def median_qoq(series):
    quarter_growths = [series[i] / series[i - 1] - 1 for i in range(1, len(series))]
    return sorted(quarter_growths)[len(quarter_growths) // 2]


def count_anchors(component, target):
    return len(
        {
            d["name"]
            for d in DATASOURCES
            if target in d["targets"]
            and component in d["component"]
            and d["role"] in ANCHOR_ROLES
        }
    )


def build_estimates(component, level, median_growth, scenario, target):
    family_estimates = {
        "trend": level * (1 + median_growth),
        "supply": level * (1 + target["supply_qoq"][component][scenario]),
        "demand": level * (1 + target["demand_qoq"][component][scenario]),
        "macro": level * (1 + median_growth) * target["macro_scaler"][scenario],
    }
    family_estimates["price"] = (
        level
        * (1 + target["volume_qoq"][scenario])
        * (1 + target["price_qoq"][scenario])
        if component == "Memory"
        else family_estimates["trend"]
    )
    if target.get("analyst_qoq"):
        family_estimates["analyst"] = level * (
            1 + target["analyst_qoq"][component][scenario]
        )
    family_estimates["reconciled"] = sum(
        target["weights"][family] * family_estimates[family]
        for family in target["weights"]
    )
    return family_estimates


def classify_confidence(anchor_count, dispersion_cv):
    if anchor_count >= CONF_HIGH["min_anchors"] and dispersion_cv < CONF_HIGH["max_cv"]:
        label = "high"
    elif anchor_count >= CONF_MED["min_anchors"] and dispersion_cv < CONF_MED["max_cv"]:
        label = "medium"
    else:
        label = "low"
    return label, round(
        min(1.0, anchor_count / 3) * max(0.0, 1 - dispersion_cv / CV_REF), 2
    )


def run_target(target_name, base_levels, relative_prior=0.0):
    target = TARGETS[target_name]
    rows, detail_rows = [], []
    for component in COMPONENTS:
        median_growth = median_qoq(history[component].to_list())
        level = base_levels[component]
        base_est = build_estimates(component, level, median_growth, "base", target)
        low_est = build_estimates(component, level, median_growth, "low", target)
        high_est = build_estimates(component, level, median_growth, "high", target)
        reconciled_base = base_est["reconciled"]
        families = [family for family in base_est if family != "reconciled"]
        spread = [
            estimate[family]
            for estimate in (low_est, base_est, high_est)
            for family in families
        ]
        dispersion_cv = pstdev(spread) / mean(spread) if mean(spread) else 0.0
        anchor_count = count_anchors(component, target_name)
        half_width = (
            dispersion_cv**2 + (BASE_HW / (anchor_count + 1) ** 0.5) ** 2
        ) ** 0.5
        if relative_prior:
            half_width = (half_width**2 + relative_prior**2) ** 0.5
        label, score = classify_confidence(anchor_count, dispersion_cv)
        rows.append(
            dict(
                component=component,
                reconciled_base=round(reconciled_base, 3),
                low=round(reconciled_base * (1 - half_width), 3),
                high=round(reconciled_base * (1 + half_width), 3),
                trend_only=round(base_est["trend"], 3),
                confidence=label,
                confidence_score=score,
                n_anchors=anchor_count,
                dispersion_cv=round(dispersion_cv, 3),
            )
        )
        for family in [
            name
            for name in [
                "trend",
                "supply",
                "demand",
                "price",
                "macro",
                "analyst",
                "reconciled",
            ]
            if name in base_est
        ]:
            detail_rows.append(
                dict(
                    component=component,
                    estimate=family,
                    value_b=round(base_est[family], 3),
                )
            )
    result = pl.DataFrame(rows)
    total_row = {"component": "TOTAL", "confidence": "-"}
    for column in ["reconciled_base", "low", "high", "trend_only"]:
        total_row[column] = round(result[column].sum(), 3)
    return pl.concat(
        [result, pl.DataFrame([total_row])], how="diagonal_relaxed"
    ), pl.DataFrame(detail_rows)

## 4. Datapoint Forecast  + sanity check vs v1

In [7]:
q4_2025_levels = {
    component: history.filter(pl.col("Quarter") == "Q4 2025")[component][0]
    for component in COMPONENTS
}
q1_result, q1_detail = run_target("Q1 2026", q4_2025_levels)
q1_total = q1_result.filter(pl.col("component") == "TOTAL").row(0, named=True)
q1_bases = {
    row["component"]: row["reconciled_base"]
    for row in q1_result.filter(pl.col("component") != "TOTAL").iter_rows(named=True)
}
q1_relative_uncertainty = ((q1_total["high"] - q1_total["low"]) / 2) / q1_total[
    "reconciled_base"
]
q2_result, q2_detail = run_target("Q2 2026", q1_bases, q1_relative_uncertainty)
q2_total = q2_result.filter(pl.col("component") == "TOTAL").row(0, named=True)
print(
    f"Q1 2026 nowcast  total: ${q1_total['reconciled_base']:.1f}B  (low ${q1_total['low']:.1f} - high ${q1_total['high']:.1f}B)"
)
print(
    f"Q2 2026 forecast total: ${q2_total['reconciled_base']:.1f}B  (low ${q2_total['low']:.1f} - high ${q2_total['high']:.1f}B)  (+{(q2_total['reconciled_base']/q1_total['reconciled_base']-1)*100:.0f}% QoQ)"
)

# Sanity check: must reproduce v1 (chip_component_spend.ipynb) within rounding.
EXPECTED = {
    "Q1 2026": {
        "TOTAL": (26.017, 21.533, 30.501),
        "Memory": 18.326,
        "Logic": 2.761,
        "Packaging": 3.106,
        "Auxiliary": 1.824,
    },
    "Q2 2026": {
        "TOTAL": (37.412, 29.164, 45.661),
        "Memory": 28.256,
        "Logic": 3.286,
        "Packaging": 3.718,
        "Auxiliary": 2.152,
    },
}


def check(result, horizon):
    expected = EXPECTED[horizon]
    total = result.filter(pl.col("component") == "TOTAL").row(0, named=True)
    assert abs(total["reconciled_base"] - expected["TOTAL"][0]) < 0.01, (
        horizon,
        "total",
        total["reconciled_base"],
    )
    assert (
        abs(total["low"] - expected["TOTAL"][1]) < 0.01
        and abs(total["high"] - expected["TOTAL"][2]) < 0.01
    )
    for row in result.filter(pl.col("component") != "TOTAL").iter_rows(named=True):
        assert abs(row["reconciled_base"] - expected[row["component"]]) < 0.01, (
            horizon,
            row["component"],
            row["reconciled_base"],
        )


check(q1_result, "Q1 2026")
check(q2_result, "Q2 2026")
print("SANITY CHECK PASSED: v2 reproduces v1 exactly (totals + every component).")
q1_result

Q1 2026 nowcast  total: $26.0B  (low $21.5 - high $30.5B)
Q2 2026 forecast total: $37.4B  (low $29.2 - high $45.7B)  (+44% QoQ)
SANITY CHECK PASSED: v2 reproduces v1 exactly (totals + every component).


component,reconciled_base,low,high,trend_only,confidence,confidence_score,n_anchors,dispersion_cv
str,f64,f64,f64,f64,str,f64,i64,f64
"""Memory""",18.326,14.355,22.297,15.127,"""medium""",0.4,5,0.211
"""Logic""",2.761,2.584,2.939,2.808,"""high""",0.9,4,0.036
"""Packaging""",3.106,2.889,3.322,3.206,"""high""",0.87,4,0.044
"""Auxiliary""",1.824,1.705,1.943,1.868,"""high""",0.89,4,0.037
"""TOTAL""",26.017,21.533,30.501,23.009,"""-""",null,null,null


In [8]:
q2_result

component,reconciled_base,low,high,trend_only,confidence,confidence_score,n_anchors,dispersion_cv
str,f64,f64,f64,f64,str,f64,i64,f64
"""Memory""",28.256,21.725,34.787,25.201,"""high""",0.58,6,0.147
"""Logic""",3.286,2.66,3.912,3.458,"""high""",0.83,4,0.061
"""Packaging""",3.718,3.024,4.413,3.873,"""high""",0.85,5,0.053
"""Auxiliary""",2.152,1.755,2.549,2.224,"""high""",0.87,6,0.047
"""TOTAL""",37.412,29.164,45.661,34.756,"""-""",null,null,null


## 5. Appropriate Visualization

In [9]:
figure = go.Figure()
figure.add_scatter(
    x=history["Quarter"].to_list(),
    y=history["Total"].to_list(),
    mode="lines+markers",
    name="history (actual)",
)
for label, total, color in [
    ("Q1 2026 nowcast", q1_total, "crimson"),
    ("Q2 2026 forecast", q2_total, "darkorange"),
]:
    figure.add_scatter(
        x=[label[:7]],
        y=[total["reconciled_base"]],
        mode="markers",
        name=label,
        marker=dict(size=11, color=color),
        error_y=dict(
            type="data",
            symmetric=False,
            array=[total["high"] - total["reconciled_base"]],
            arrayminus=[total["reconciled_base"] - total["low"]],
        ),
    )
figure.update_layout(
    title="Chip component spend: history + Q1 nowcast + Q2 forecast",
    yaxis_title="USD billions",
    height=480,
)
figure

In [10]:
history_long = history.unpivot(
    index="Quarter", on=COMPONENTS, variable_name="Component", value_name="spend"
)


def reconciled_rows(result, quarter_name):
    return result.filter(pl.col("component") != "TOTAL").select(
        Quarter=pl.lit(quarter_name),
        Component=pl.col("component"),
        spend=pl.col("reconciled_base"),
    )


breakdown_bars = pl.concat(
    [
        history_long.select(["Quarter", "Component", "spend"]),
        reconciled_rows(q1_result, "Q1 2026"),
        reconciled_rows(q2_result, "Q2 2026"),
    ],
    how="vertical_relaxed",
)
quarter_order = history["Quarter"].to_list() + ["Q1 2026", "Q2 2026"]
figure = px.bar(
    breakdown_bars,
    x="Quarter",
    y="spend",
    color="Component",
    category_orders={"Quarter": quarter_order, "Component": COMPONENTS},
    color_discrete_map=COLORS,
    labels={"spend": "USD billions"},
    title="Component breakdown: history -> Q1 nowcast -> Q2 forecast",
)
figure.add_annotation(
    x="Q1 2026",
    y=q1_total["reconciled_base"],
    text="nowcast",
    showarrow=True,
    arrowhead=2,
    yshift=8,
)
figure.add_annotation(
    x="Q2 2026",
    y=q2_total["reconciled_base"],
    text="forecast",
    showarrow=True,
    arrowhead=2,
    yshift=8,
)
figure.update_layout(height=540)
figure

In [11]:
family_order = ["trend", "supply", "demand", "price", "macro", "analyst", "reconciled"]
figure = px.bar(
    q2_detail.to_pandas(),
    x="estimate",
    y="value_b",
    facet_col="component",
    color="estimate",
    category_orders={"estimate": family_order, "component": COMPONENTS},
    labels={"value_b": "USD billions"},
    title="Q2 2026: estimate by family, per component",
)
figure.update_yaxes(matches=None)
figure.update_layout(height=430, showlegend=False)
figure

## Summary
- Inputs are read entirely from `data/records/` (1 processed_data history card + 23 news/facts datasource records); v1's `datasources_old/` is never touched.
- The model/engine is byte-for-byte v1; the **sanity check in §4 asserts the forecast matches v1** (Q1 $26.0B, Q2 $37.4B, and every component). The one deliberate difference — FRED is read from a pinned `facts` record instead of fetched live — is what makes v2 deterministic and is documented in that record's methodology.
- §6 writes the forecast back into the store as `forecasts` records, with `methodology.inputs` pointing at the records used and `adjacency` backfilled.

## 6. Persist forecast to the store
The notebook *is* the agent here: it writes the per-horizon estimate tables as `processed_data`
blobs+cards, then two `forecasts` records, and backfills the `adjacency` of every input. Deterministic
ids → re-execution overwrites in place. Lineage: `history → q*-estimates → forecast`, Q2 chaining off Q1.

In [12]:
RECORDS, BLOBS = REPO / "data" / "records", REPO / "data" / "blobs"
NOW = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")
BUCKET = NOW[:7].replace("-", "/")
NOTEBOOK = "notebooks/chip_component_spend_v2.ipynb"
QUARTER = {"Q1 2026": "2026-Q1", "Q2 2026": "2026-Q2"}

# id -> path is known from the explicit manifest (§0) + records this cell writes; no scanning.
record_paths = {history_card["id"]: REPO / HISTORY_RECORD}
for relative_path, record in zip(DATASOURCE_RECORDS, datasource_records):
    record_paths[record["id"]] = REPO / relative_path


def add_adjacency(upstream_id, downstream_id):
    path = record_paths[upstream_id]
    rec = json.loads(path.read_text())
    if downstream_id not in rec["adjacency"]:
        rec["adjacency"].append(downstream_id)
        path.write_text(json.dumps(rec, indent=2) + "\n")


def write_record(rec):
    bucketed = rec["type"] in {"facts", "forecasts", "computations", "news"}
    folder = RECORDS / rec["type"] / BUCKET if bucketed else RECORDS / rec["type"]
    folder.mkdir(parents=True, exist_ok=True)
    path = folder / (rec["id"] + ".json")
    path.write_text(json.dumps(rec, indent=2) + "\n")
    record_paths[rec["id"]] = path
    for upstream_id in rec["methodology"]["inputs"]:
        add_adjacency(upstream_id, rec["id"])
    return rec["id"]


history_id = history_card["id"]
datasource_ids = [r["id"] for r in datasource_records]


def estimate_card(horizon, result, blob_name, card_id, inputs):
    result.write_csv(BLOBS / blob_name)
    return {
        "id": card_id,
        "type": "processed_data",
        "description": f"Chip component spend {horizon} per-component estimate table (reconciled base, low/high band, confidence) from v2.",
        "tags": [
            "chip-component-spend",
            "estimates",
            QUARTER[horizon].lower(),
            "processed",
        ],
        "ts_recorded": NOW,
        "ts_applies": QUARTER[horizon],
        "confidence": 0.6,
        "source": f"notebook:{NOTEBOOK}",
        "body": {
            "blob_path": f"data/blobs/{blob_name}",
            "format": "csv",
            "columns": result.columns,
            "row_count": result.height,
            "units": {
                "reconciled_base": "USD billions",
                "low": "USD billions",
                "high": "USD billions",
                "trend_only": "USD billions",
            },
            "how_to_read": "One row per component plus a TOTAL row; reconciled_base is the central estimate, low/high the band.",
        },
        "adjacency": [],
        "methodology": {
            "summary": f"v2 engine output for {horizon}: reconciled weighted blend of estimate families with a data-driven band.",
            "inputs": inputs,
            "code": NOTEBOOK,
        },
    }


q1_card_id, q2_card_id = (
    "pd_chip-spend-q1-estimates_ea34",
    "pd_chip-spend-q2-estimates_1f45",
)
write_record(
    estimate_card(
        "Q1 2026",
        q1_result,
        "chip_component_spend_q1_2026_estimates.csv",
        q1_card_id,
        [history_id] + datasource_ids,
    )
)
write_record(
    estimate_card(
        "Q2 2026",
        q2_result,
        "chip_component_spend_q2_2026_estimates.csv",
        q2_card_id,
        [history_id, q1_card_id] + datasource_ids,
    )
)


def forecast_record(horizon, kind, result, fc_id, inputs):
    components, weighted, total_base = {}, 0.0, 0.0
    for row in result.filter(pl.col("component") != "TOTAL").iter_rows(named=True):
        components[row["component"]] = {
            "base": row["reconciled_base"],
            "low": row["low"],
            "high": row["high"],
            "confidence": row["confidence"],
            "confidence_score": row["confidence_score"],
        }
        weighted += row["reconciled_base"] * row["confidence_score"]
        total_base += row["reconciled_base"]
    total = result.filter(pl.col("component") == "TOTAL").row(0, named=True)
    confidence = round(weighted / total_base, 2)
    return {
        "id": fc_id,
        "type": "forecasts",
        "description": f"Chip component spend {kind} for {horizon}: ${total['reconciled_base']:.1f}B (NVIDIA/AMD/Google/Amazon), band ${total['low']:.1f}-{total['high']:.1f}B.",
        "tags": ["chip-component-spend", "forecast", kind, QUARTER[horizon].lower()],
        "ts_recorded": NOW,
        "ts_applies": QUARTER[horizon],
        "confidence": confidence,
        "source": f"notebook:{NOTEBOOK}",
        "body": {
            "metric": "AI-chip component spend (NVIDIA/AMD/Google/Amazon)",
            "kind": kind,
            "horizon": QUARTER[horizon],
            "unit": "USD billions",
            "total": {
                "base": total["reconciled_base"],
                "low": total["low"],
                "high": total["high"],
            },
            "components": components,
        },
        "adjacency": [],
        "methodology": {
            "summary": f"v2 reconcile-and-band engine ({kind} for {horizon}). Confidence {confidence} = spend-weighted mean of per-component confidence scores. Q2 chains off the Q1 nowcast (uncertainty added in quadrature).",
            "inputs": inputs,
            "code": NOTEBOOK,
        },
    }


fc_q1_id, fc_q2_id = (
    "fc_chip-component-spend-q1-2026_3c7d",
    "fc_chip-component-spend-q2-2026_9f2a",
)
write_record(forecast_record("Q1 2026", "nowcast", q1_result, fc_q1_id, [q1_card_id]))
write_record(
    forecast_record("Q2 2026", "forecast", q2_result, fc_q2_id, [q2_card_id, fc_q1_id])
)
print("wrote estimate cards:", q1_card_id, q2_card_id)
print("wrote forecast records:", fc_q1_id, fc_q2_id)
print(json.dumps(json.loads(record_paths[fc_q1_id].read_text()), indent=2))

wrote estimate cards: pd_chip-spend-q1-estimates_ea34 pd_chip-spend-q2-estimates_1f45
wrote forecast records: fc_chip-component-spend-q1-2026_3c7d fc_chip-component-spend-q2-2026_9f2a
{
  "id": "fc_chip-component-spend-q1-2026_3c7d",
  "type": "forecasts",
  "description": "Chip component spend nowcast for Q1 2026: $26.0B (NVIDIA/AMD/Google/Amazon), band $21.5-30.5B.",
  "tags": [
    "chip-component-spend",
    "forecast",
    "nowcast",
    "2026-q1"
  ],
  "ts_recorded": "2026-06-12T11:28:09Z",
  "ts_applies": "2026-Q1",
  "confidence": 0.54,
  "source": "notebook:notebooks/chip_component_spend_v2.ipynb",
  "body": {
    "metric": "AI-chip component spend (NVIDIA/AMD/Google/Amazon)",
    "kind": "nowcast",
    "horizon": "2026-Q1",
    "unit": "USD billions",
    "total": {
      "base": 26.017,
      "low": 21.533,
      "high": 30.501
    },
    "components": {
      "Memory": {
        "base": 18.326,
        "low": 14.355,
        "high": 22.297,
        "confidence": "medium",
